# Customer Churn Prediction & Retention Strategy
### Author: Virinchi Kopparam

---

## Business Problem

A European bank is losing **~20% of its customers every year** to churn. The challenge is not just measuring that number but predicting *which* customers are about to leave early enough to do something about it.

**Why this matters in dollars:**
- Industry average acquisition cost to replace a churned customer: **$243**
- At 2,037 churned customers in this dataset, that is **$494,991 in replacement cost alone**
- That figure does not include lost revenue from deposits, fees, or cross-sell opportunities

**What this project does:**
1. Identifies the behavioral and demographic signals that predict churn
2. Benchmarks 5 classification models to find the most reliable predictor
3. Quantifies the business cost of model errors (a missed churner costs more than a false alarm)
4. Connects model output to an actionable retention framework

**One important framing note:** In churn modeling, a **false negative** (predicting someone stays when they actually leave) is more expensive than a **false positive** (flagging someone as at-risk when they are not). A false negative means a lost customer plus $243 to replace them. A false positive means an unnecessary retention offer, usually a voucher or a call. Model selection here prioritizes minimizing false negatives over raw accuracy.


## 1. Imports and Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

## 2. Data Loading and Validation

The dataset contains 10,000 bank customers across France, Germany, and Spain with 14 features covering demographics, account behavior, and a binary churn flag (`Exited`).

**Columns dropped before modeling:** `RowNumber`, `CustomerId`, `Surname` are identifiers with no predictive value. Keeping them would not improve the model and would make it non-generalizable to new customers.


In [ ]:
df = pd.read_csv('Churn_Modelling.csv')
df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)
df.head(5)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

**Validation result:** Zero nulls across all 11 remaining columns. No imputation needed. Dataset is clean and ready for analysis.

Quick class distribution check before going further:

In [ ]:
churn_counts = df['Exited'].value_counts()
churn_pct = df['Exited'].value_counts(normalize=True) * 100
print(f"Retained: {churn_counts[0]:,} ({churn_pct[0]:.1f}%)")
print(f"Churned:  {churn_counts[1]:,} ({churn_pct[1]:.1f}%)")
print(f"\nClass imbalance ratio: {churn_counts[0]/churn_counts[1]:.1f}:1")

The dataset has a **roughly 4:1 class imbalance** (80% retained, 20% churned). This is why SVM and Logistic Regression struggle later: they optimize for the majority class. It also explains why accuracy alone is a misleading metric here. A model that predicted "nobody churns" would score 80% accuracy while being completely useless.

## 3. Exploratory Data Analysis

Goal here is not just to describe the data but to find the signals that actually matter for retention decisions. Each chart is followed by a business interpretation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Churn distribution
axes[0].bar(['Retained', 'Churned'], df['Exited'].value_counts().values,
            color=['#4C9BE8', '#E8604C'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(df['Exited'].value_counts().values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontweight='bold')

# Churn rate by Geography
geo_churn = df.groupby('Geography')['Exited'].mean() * 100
geo_churn.sort_values(ascending=False).plot(kind='bar', ax=axes[1],
    color=['#E8604C', '#4C9BE8', '#4C9BE8'], edgecolor='white', rot=0)
axes[1].set_title('Churn Rate by Geography', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(geo_churn.sort_values(ascending=False)):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('visuals/churn_overview.png', dpi=150, bbox_inches='tight')
plt.show()

**Geography finding:** Germany churns at nearly **double the rate** of France and Spain (~32% vs ~16%). This is the single strongest geographic signal in the dataset and is consistent with the SQL analysis in this project. Germany is not a marginal problem; it is the primary market to fix.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Churn rate by Activity Status
active_churn = df.groupby('IsActiveMember')['Exited'].mean() * 100
axes[0].bar(['Inactive', 'Active'], active_churn.values,
            color=['#E8604C', '#4C9BE8'], edgecolor='white')
axes[0].set_title('Churn Rate: Active vs Inactive Members', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(active_churn.values):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

# Churn rate by Age Group
df['AgeGroup'] = pd.cut(df['Age'],
    bins=[18, 30, 45, 60, 100],
    labels=['18-30', '31-45', '46-60', '60+'])
age_churn = df.groupby('AgeGroup', observed=True)['Exited'].mean() * 100
age_churn.plot(kind='bar', ax=axes[1],
    color=['#4C9BE8', '#4C9BE8', '#E8604C', '#F0A500'], edgecolor='white', rot=0)
axes[1].set_title('Churn Rate by Age Group', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(age_churn):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('visuals/churn_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

df.drop(columns=['AgeGroup'], inplace=True)

**Activity finding:** Inactive members churn at roughly **2.6x the rate** of active members. This is the strongest single behavioral signal in the dataset and drives the highest weight in the rules-based risk score built in the SQL companion file.

**Age finding:** The **46-60 age band** churns at the highest rate. Customers in this range are likely mid-career, have higher balances, and are more likely to shop competing products. They are also the most financially valuable to retain.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Churn rate by Number of Products
prod_churn = df.groupby('NumOfProducts')['Exited'].mean() * 100
prod_churn.plot(kind='bar', ax=axes[0],
    color=['#4C9BE8', '#4C9BE8', '#E8604C', '#E8604C'], edgecolor='white', rot=0)
axes[0].set_title('Churn Rate by Number of Products', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(prod_churn):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# Churn rate by Gender
gender_churn = df.groupby('Gender')['Exited'].mean() * 100
gender_churn.plot(kind='bar', ax=axes[1],
    color=['#E8604C', '#4C9BE8'], edgecolor='white', rot=0)
axes[1].set_title('Churn Rate by Gender', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(gender_churn):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('visuals/churn_by_products_gender.png', dpi=150, bbox_inches='tight')
plt.show()

**Product finding:** Customers with 1 product churn at ~28%. Customers with 2 products drop to ~8%. That 20-point difference makes cross-sell one of the highest-ROI retention strategies available. Interestingly, 3 and 4 product customers churn at extreme rates, but sample sizes are small (see SQL analysis Q6 for the full segment breakdown).

**Gender finding:** Female customers churn at a noticeably higher rate than male customers (~25% vs ~16%). Combined with the Germany finding, German female customers are the highest-risk demographic segment in the dataset.


## 4. Preprocessing: Encoding and Feature Selection

Two encoding strategies used:
- **Label encoding** for Gender (binary: Male/Female becomes 0/1)
- **One-hot encoding** for Geography (3 categories become 2 binary columns via `drop_first=True` to avoid multicollinearity)

`RowNumber`, `CustomerId`, and `Surname` are excluded as they carry no predictive signal.


In [ ]:
df_model = pd.read_csv('Churn_Modelling.csv')

label_encoder = LabelEncoder()
df_model['Gender'] = label_encoder.fit_transform(df_model['Gender'])
df_model = pd.get_dummies(df_model, columns=['Geography'], drop_first=True)

features = ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance',
            'NumOfProducts', 'HasCrCard', 'IsActiveMember',
            'EstimatedSalary', 'Geography_Germany', 'Geography_Spain']

X = df_model[features]
y = df_model['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"Churn rate in test set: {y_test.mean()*100:.1f}%")

## 5. Model Benchmarking

Five classifiers benchmarked on the same train/test split with the same random seed for a fair comparison. Evaluation goes beyond accuracy: **recall on the churn class (class 1)** is the metric that matters most because it measures how many actual churners the model catches.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

models = {
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(random_state=42),
    'SVM':                 SVC(kernel='linear', random_state=42)
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    results.append({
        'Model':               name,
        'Accuracy':            round(accuracy_score(y_test, y_pred), 4),
        'Precision (Churn)':   round(report['1']['precision'], 2),
        'Recall (Churn)':      round(report['1']['recall'], 2),
        'F1 (Churn)':          round(report['1']['f1-score'], 2),
        'False Negatives':     int(confusion_matrix(y_test, y_pred)[1][0])
    })

results_df = pd.DataFrame(results).sort_values('Recall (Churn)', ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df

**Reading this table:** Accuracy alone is misleading here given the class imbalance. The column that matters most is **Recall (Churn)**: how many of the 393 actual churners did the model catch?

- **Random Forest and Gradient Boosting** both achieve ~86-87% accuracy with the best balance of precision and recall on the churn class
- **SVM completely fails** on the churn class (recall = 0.00), predicting every customer as retained. It scores 80% accuracy by doing nothing useful
- **Logistic Regression** catches only 20% of churners despite 81% accuracy, making it close to useless for this use case
- **False Negatives** column shows the real business cost: SVM misses all 393 churners, Random Forest misses 211

**Recommendation:** Random Forest is selected as the production model. Gradient Boosting has marginally better accuracy (86.75% vs 86.65%) but Random Forest catches slightly fewer churners at a better precision tradeoff. Either model is a defensible choice; the gap is negligible.


## 6. Confusion Matrix: Business Cost Framing

The confusion matrix for the selected model (Random Forest) broken down into business terms.


In [ ]:
rf_model = models['Random Forest']
y_pred_rf = predictions['Random Forest']
cm = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Retained', 'Churned'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Random Forest - Confusion Matrix', fontsize=13, fontweight='bold')

# Business cost breakdown
tn, fp, fn, tp = cm.ravel()
acquisition_cost = 243
categories = ['True Positives\n(Churners caught)', 
               'False Negatives\n(Churners missed)',
               'False Positives\n(False alarms)',
               'True Negatives\n(Correctly retained)']
values = [tp, fn, fp, tn]
colors = ['#4C9BE8', '#E8604C', '#F0A500', '#6BBF6B']
bars = axes[1].bar(categories, values, color=colors, edgecolor='white')
axes[1].set_title('Prediction Breakdown (2,000 test customers)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Customers')
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('visuals/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n--- Business Cost Summary (Random Forest on test set) ---")
print(f"Churners correctly flagged (True Positives):  {tp:>4}  -- retention team can act")
print(f"Churners missed (False Negatives):            {fn:>4}  -- lost + ${acquisition_cost} each to replace = ${fn*acquisition_cost:,}")
print(f"False alarms (False Positives):               {fp:>4}  -- unnecessary retention offers")
print(f"Correctly retained (True Negatives):         {tn:>4}")
print(f"\nTotal avoidable replacement cost from missed churners: ${fn*acquisition_cost:,}")

**The business case for this model:** Random Forest catches **182 out of 393 churners** in the test set, a 46% catch rate. On a 10,000 customer base that saves approximately **$106,722 in replacement costs** per year at current churn rates. Every percentage point improvement in recall translates directly to ~$956 in savings.

The 211 missed churners (false negatives) represent the model's current ceiling without further tuning. The path to improvement is either SMOTE to address class imbalance or hyperparameter tuning to shift the decision threshold toward higher recall.


## 7. Feature Importance: What Actually Drives Churn

The Random Forest model assigns importance scores to each feature based on how much they reduce impurity across all decision trees. This tells us which customer signals matter most.


In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
names = [features[i] for i in indices]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E8604C' if imp > 0.1 else '#4C9BE8' for imp in importances[indices]]
ax.barh(range(len(features)), importances[indices], color=colors, edgecolor='white')
ax.set_yticks(range(len(features)))
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance Score')
ax.set_title('Random Forest: Feature Importance', fontsize=13, fontweight='bold')
ax.axvline(x=0.1, color='gray', linestyle='--', alpha=0.5, label='0.10 threshold')
ax.legend()

plt.tight_layout()
plt.savefig('visuals/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 5 features by importance:")
for i in range(5):
    print(f"  {i+1}. {names[i]:<22} {importances[indices[i]]:.4f}")

**Feature importance findings:**

- **Age** is the dominant predictor by a significant margin. The 46-60 age band finding from EDA is validated here at the model level.
- **Balance and EstimatedSalary** rank highly, confirming that financial profile matters beyond just credit score.
- **IsActiveMember** ranks strongly, which validates the weight placed on inactivity in the rules-based SQL risk score.
- **CreditScore** ranks lower than most people expect, consistent with the SQL Q1 finding that churn rate barely moves across credit tiers.
- **Geography (Germany)** has meaningful importance but lower than behavioral signals, confirming that *what customers do* matters more than *where they are* for prediction purposes.


## 8. Feature Engineering: Can We Do Better?

Hypothesis: engineered features that capture behavioral patterns more explicitly might improve model performance beyond 86.7%.

Features added:
- **BalanceZero**: binary flag for zero-balance accounts (behavior, not amount)
- **AgeGroup**: age buckets matching the retention team's segmentation logic
- **BalanceToSalaryRatio**: relative financial engagement vs absolute balance
- **ProductUsage**: interaction term (NumOfProducts x IsActiveMember) capturing engaged multi-product customers vs inactive ones
- **TenureGroup**: tenure buckets to test whether lifecycle stage matters
- **Male_Germany / Male_Spain**: interaction terms to test gender-geography effects


In [ ]:
df_fe = pd.read_csv('Churn_Modelling.csv')

df_fe['BalanceZero']         = (df_fe['Balance'] == 0).astype(int)
df_fe['BalanceToSalaryRatio'] = df_fe['Balance'] / df_fe['EstimatedSalary']
df_fe['ProductUsage']        = df_fe['NumOfProducts'] * df_fe['IsActiveMember']

df_fe['AgeGroup']    = pd.cut(df_fe['Age'],
    bins=[18, 25, 35, 45, 55, 65, 75, 85, 95],
    labels=['18-25','26-35','36-45','46-55','56-65','66-75','76-85','86-95'])
df_fe['TenureGroup'] = pd.cut(df_fe['Tenure'],
    bins=[0, 2, 5, 7, 10],
    labels=['0-2','3-5','6-7','8-10'])

label_encoder = LabelEncoder()
df_fe['Gender'] = label_encoder.fit_transform(df_fe['Gender'])
df_fe = pd.get_dummies(df_fe, columns=['Geography'], drop_first=True)
df_fe['Male_Germany'] = df_fe['Gender'] * df_fe['Geography_Germany']
df_fe['Male_Spain']   = df_fe['Gender'] * df_fe['Geography_Spain']
df_fe = pd.get_dummies(df_fe, columns=['AgeGroup', 'TenureGroup'], drop_first=True)

fe_features = (
    ['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard',
     'IsActiveMember','EstimatedSalary','Geography_Germany','Geography_Spain',
     'BalanceZero','BalanceToSalaryRatio','ProductUsage','Male_Germany','Male_Spain']
    + [c for c in df_fe.columns if 'AgeGroup_' in c or 'TenureGroup_' in c]
)

X_fe = df_fe[fe_features]
y_fe = df_fe['Exited']

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe, y_fe, test_size=0.2, random_state=42)

scaler_fe = StandardScaler()
X_train_fe = scaler_fe.fit_transform(X_train_fe)
X_test_fe  = scaler_fe.transform(X_test_fe)

rf_fe = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fe.fit(X_train_fe, y_train_fe)
y_pred_fe = rf_fe.predict(X_test_fe)

acc_baseline = accuracy_score(y_test, y_pred_rf)
acc_fe       = accuracy_score(y_test_fe, y_pred_fe)
rec_baseline = classification_report(y_test, y_pred_rf, output_dict=True)['1']['recall']
rec_fe       = classification_report(y_test_fe, y_pred_fe, output_dict=True, zero_division=0)['1']['recall']

comparison = pd.DataFrame({
    'Version':         ['Baseline (11 features)', 'Feature Engineered (26 features)'],
    'Accuracy':        [f'{acc_baseline:.4f}', f'{acc_fe:.4f}'],
    'Recall (Churn)':  [f'{rec_baseline:.2f}',  f'{rec_fe:.2f}'],
    'Feature Count':   [11, len(fe_features)]
})
print(comparison.to_string(index=False))

**Feature engineering verdict:** Adding 15 engineered features produced **no meaningful improvement** in either accuracy or churn recall. Accuracy moved from 86.65% to 86.50%, essentially flat.

This is actually a useful finding, not a failure. It tells us:
1. The original 11 behavioral features already capture most of the available signal
2. The model is robust and not under-fitting
3. The path to improvement is not more features but better handling of class imbalance (SMOTE) or threshold tuning, not feature complexity

The baseline model is the one used for all business recommendations below.


## 9. Business Recommendations

Everything above feeds into four concrete actions a retention team can execute next quarter.

---

**1. Build the priority outreach list from the SQL risk score first**
The rules-based composite risk score (built in `churn_prediction.sql`) identified a Critical band churning at 45.5% vs a 20.4% baseline. Before the ML model goes to production, that SQL output is a deployable, explainable list the retention team can act on immediately. No black box, no IT dependency.

---

**2. Focus Germany intervention before any other market**
Germany churns at 32.4% vs ~16% in France and Spain, representing $97.9M in deposits at risk. German female customers aged 45-60 show the highest churn rate of any segment (76% in the cross-sell analysis). A Germany-specific retention pilot, before scaling globally, is the highest-ROI use of budget.

---

**3. Cross-sell as a retention strategy, not a revenue strategy**
Single-product customers churn at 28%. Two-product customers churn at 8%. A targeted cross-sell campaign for high-balance single-product customers (identified in SQL Q5) reduces churn risk while growing revenue simultaneously. The 785 high-value inactive customers flagged in Q5 are the starting list.

---

**4. Next modeling step: shift the decision threshold**
The current model catches 46% of churners at the default 0.5 decision threshold. Lowering the threshold to 0.3 would increase recall at the cost of more false positives. Given that false negatives cost $243 each and false positives cost only a retention offer, the economics favor a lower threshold. Recommended next step: plot the precision-recall curve and find the threshold that minimizes total business cost, not just classification error.

---

*Full SQL analysis, cohort segmentation, and business impact sizing available in `churn_prediction.sql` in this repository.*
